# IE4483 Sentiment Analysis: all models in one notebook

This notebook covers the whole pipeline: preprocessing, four classifiers, accuracy on a common validation split, and 5-fold cross-validation for every model.

| # | Model | Features | Classifier |
|---|---|---|---|
| 1 | **LR default** (Part 1) | default `TfidfVectorizer()`, as in `tfidf_features_example` | logistic regression, C=1 |
| 2 | **LR tuned** | word 1–2-grams + character 2–5-grams TF-IDF, sublinear tf | logistic regression, C=4, balanced class weights (chosen by 5-fold CV) |
| 3 | **BiLSTM A: scratch** | token ids, embeddings learned from `train.json` only | our BiLSTM + attention network |
| 4 | **BiLSTM B: GloVe** | token ids, embeddings initialised from GloVe-100d, then fine-tuned | same network |

`test.json` has **no labels**, so all accuracies are measured on labelled data held out from `train.json`. The test set is only predicted, for `submission.csv`.

**Contents:** 0. How to run · 1. Data · 2. Preprocessing for TF-IDF · 3. Model 1: LR default · 4. Model 2: LR tuned · 5. Preprocessing for the BiLSTM · 6. BiLSTM architecture · 7. Loss and training strategy · 8. BiLSTM hyper-parameter selection · 9. Reproducibility (3 seeds) · 10. Hold-out comparison of all models · 11. **5-fold cross-validation of all models** · 12. Error analysis · 13. Test predictions · 14. Summary

## 0. How to run
1. Open this notebook from the `IE4483 Project` folder and select the kernel **"Python (IE4483 sentiment_project)"** (the project's `.venv`: Python, PyTorch, scikit-learn, pandas, matplotlib).
2. GloVe vectors (`glove-wiki-gigaword-100`, 128 MB) are downloaded automatically to `~/glove/` the first time.
3. Run all cells. Every BiLSTM training run is cached in `../../results/bilstm_runs.pkl`, so a second run takes a few minutes. Set `RUN_ALL = True` in the next cell to retrain everything from scratch (about 45 minutes on an Apple M4 Pro CPU). The logistic regression models train in seconds and are always retrained.

Training runs on the **CPU** on purpose. On this machine it was faster than the Apple GPU (MPS) for this small LSTM (about 19 s vs 33 s per epoch in a benchmark). CPU results are also bit-for-bit reproducible for a fixed seed; section 9 checks this.

In [ ]:
import sys, time, pickle, json, gzip, urllib.request, collections
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

sys.path.insert(0, 'Packages')                         # the project's helpers (Code/Packages)
from common import RESULTS, SEED, SEEDS, set_seed     # SEED=42, SEEDS=[42, 43, 44]
from data_loader import load_train, load_test
from preprocessing import tokenize
from splits import holdout, get_folds
from evaluate import metrics
from train_tfidf import featurise, make_clf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve

RUN_ALL = False                                        # True = ignore the cache and retrain every run
DEVICE = torch.device('cpu')
CACHE = RESULTS / 'bilstm_runs.pkl'
GLOVE = Path.home() / 'glove' / 'glove-wiki-gigaword-100.gz'

BLUE, ORANGE, AQUA, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#8a8984'
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'
COL = {'scratch': ORANGE, 'glove': BLUE}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'font.size': 10})
pd.set_option('display.max_colwidth', 110)
print('torch', torch.__version__, '| device', DEVICE, '| threads', torch.get_num_threads())

## 1. Data

**Split:** every model uses the same grouped, stratified 5-fold split of `train.json` (`splits.get_folds`):
- **stratified:** each fold keeps the 85/15 positive/negative ratio
- **grouped:** duplicate reviews (about 220) always land in the same fold, so no model is scored on a review it was trained on

**Fold 0 is the hold-out validation set** (1,480 reviews) used for training curves, hyper-parameter choices and the main comparison (section 10). All 5 folds are used for cross-validation (section 11).

In [ ]:
df, test_df = load_train(), load_test()
X_text, y = df['reviews'].values, df['sentiments'].values
folds = get_folds(df)
tr_idx, va_idx = holdout(df)
assert np.array_equal(va_idx, folds[0][1])
print('train.json:', df.shape, '| test.json:', test_df.shape, '(columns:', list(test_df.columns[1:]), '-> no labels)')
print(f'hold-out split: train part {len(tr_idx)} | validation {len(va_idx)} '
      f'({(y[va_idx] == 0).sum()} negative, {(y[va_idx] == 1).sum()} positive)')

counts = np.bincount(y)
wlen = df['reviews'].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
bars = axes[0].bar(['Negative (0)', 'Positive (1)'], counts, color=[ORANGE, BLUE], width=0.5)
for b, v in zip(bars, counts):
    axes[0].text(b.get_x() + b.get_width() / 2, v + 80, f'{v:,} ({v / counts.sum():.1%})', ha='center', color=INK)
axes[0].set_title('Class balance in train.json', loc='left', color=INK); axes[0].grid(axis='x', visible=False)
bins = np.arange(0, 305, 10)
for lab, col, name in [(1, BLUE, 'Positive'), (0, ORANGE, 'Negative')]:
    axes[1].hist(wlen[y == lab].clip(upper=300), bins=bins, histtype='step', linewidth=2, color=col, density=True, label=name)
axes[1].set_title('Review length in words (capped at 300)', loc='left', color=INK)
axes[1].set_xlabel('Words'); axes[1].set_ylabel('Share of reviews'); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print(f'Always predicting "positive" would already score {counts[1] / counts.sum():.1%} accuracy, '
      'so per-class accuracy and macro-F1 are reported alongside accuracy.')

## 2. Preprocessing for the TF-IDF models

The example notebook's `TfidfVectorizer()` defaults:
- **lowercase** the text
- **tokenise** with `(?u)\b\w\w+\b`: words of 2 or more characters. Punctuation is dropped, `"I"` and `"a"` are dropped, and `"don't"` becomes `"don"`
- **no stop-word removal**, unigrams only, every word kept
- **TF-IDF weighting:** term count × smoothed inverse document frequency, then each review vector is L2-normalised

The vectorizer is **fitted on training text only** and then reused, unchanged, to transform validation and test text.

In [ ]:
sample = "I didn't like it AT ALL!! The zipper broke after 2 days... NOT worth $20?"
print('raw:              ', sample)
print('TF-IDF tokens:    ', TfidfVectorizer().build_analyzer()(sample))

vec_full = TfidfVectorizer().fit(X_text)                       # as in the example: fit on all of train.json
print('\nX_train', vec_full.transform(X_text).shape, '| X_test', vec_full.transform(test_df['reviews']).shape,
      '(same as the example notebook)')

vec0 = TfidfVectorizer().fit(X_text[tr_idx])                   # for evaluation: fit on the training part only
Xtr0, Xva0 = vec0.transform(X_text[tr_idx]), vec0.transform(X_text[va_idx])
print(f'hold-out: X_train_part {Xtr0.shape}, X_val {Xva0.shape}; '
      f'a review uses on average {Xtr0.getnnz(axis=1).mean():.0f} of {Xtr0.shape[1]:,} words')

vocab0 = vec0.get_feature_names_out(); r = Xtr0[0]
print('\nReview 0:', X_text[tr_idx][0])
pd.DataFrame({'word': vocab0[r.indices], 'tf-idf': r.data, 'idf': vec0.idf_[r.indices]}).sort_values('tf-idf', ascending=False).round(3).head(10)

In [ ]:
# IDF: common words get low weights, rare words high weights
idf = pd.Series(vec0.idf_, index=vocab0).sort_values()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.hist(vec0.idf_, bins=40, color=BLUE, edgecolor='white', linewidth=0.5)
ax.set_title('IDF weights across the TF-IDF vocabulary', loc='left', color=INK)
ax.set_xlabel('IDF value'); ax.set_ylabel('Number of words'); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()
print('Lowest IDF (most common):', ', '.join(idf.index[:12]))
print('Highest IDF (rarest):    ', ', '.join(idf.index[-12:]))

## 3. Model 1: TF-IDF + logistic regression (default, Part 1)

Logistic regression: `p(positive) = sigmoid(w·x + b)`, with one weight per vocabulary word, trained by minimising cross-entropy with L2 regularisation (C=1).

In [ ]:
def class_accuracy(y_true, p):
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {'accuracy': (tn + tp) / len(pred), 'neg_accuracy': tn / (tn + fp), 'pos_accuracy': tp / (tp + fn),
            'macro_f1': f1_score(y_true, pred, average='macro'), 'neg_correct': f'{tn}/{tn + fp}', 'pos_correct': f'{tp}/{tp + fn}'}

def show_eval(name, p):
    a = class_accuracy(y[va_idx], p)
    print(f"{name}: accuracy {a['accuracy']:.2%} | negative reviews correct {a['neg_correct']} ({a['neg_accuracy']:.1%}) | "
          f"positive reviews correct {a['pos_correct']} ({a['pos_accuracy']:.1%}) | macro-F1 {a['macro_f1']:.4f}\n")
    print(classification_report(y[va_idx], (p >= 0.5).astype(int), target_names=['negative', 'positive'], digits=4))

t0 = time.time()
lr0 = LogisticRegression(max_iter=1000, random_state=SEED).fit(Xtr0, y[tr_idx])
p_lr0 = lr0.predict_proba(Xva0)[:, 1]; s_lr0 = time.time() - t0
show_eval('LR default', p_lr0)

coef = pd.Series(lr0.coef_[0], index=vocab0).sort_values()
top = pd.concat([coef.head(12), coef.tail(12)])
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top.index, top.values, color=[ORANGE if v < 0 else BLUE for v in top.values], height=0.65)
ax.axvline(0, color=MUTED, linewidth=1); ax.grid(axis='y', visible=False)
ax.set_title('LR default: strongest negative (orange) and positive (blue) words', loc='left', color=INK)
ax.set_xlabel('weight'); plt.tight_layout(); plt.show()

**The problem:** about 89% accuracy looks good, but the model gets **only 61 of the 216 negative reviews right (28%)**. With 85% positive training data and no class weighting, the 0.5 threshold favours "positive". Every following model is judged mainly on whether it fixes this.

## 4. Model 2: TF-IDF + logistic regression (tuned)

The project's `src/train_tfidf.py` ran a grid search with 5-fold grouped CV (192 configurations, results in `results/tfidf_cv_grid.csv`):
- features: word n-grams (1-1 or 1-2) ± character 2–5-grams, `min_df` 1 or 2
- classifier: C ∈ {0.5, 1, 2, 4, 8}, class weights none or balanced; also a linear SVM

The best logistic regression by mean CV macro-F1 is kept. Three changes fix the minority-class problem:
- **balanced class weights:** errors on negative reviews cost 5.8× more
- **bigrams** capture short phrases such as "not good" and "fell apart"
- **character n-grams** are robust to typos and word forms

In [ ]:
grid = pd.read_csv(RESULTS / 'tfidf_cv_grid.csv')
display(grid[grid.clf == 'lr'].head(8)[['clf', 'C', 'class_weight', 'ngram', 'char_wb_2_5', 'min_df',
                                        'accuracy_mean', 'recall_neg_mean', 'macro_f1_mean', 'macro_f1_std']].round(4))

bc = json.loads((RESULTS / 'best_tfidf_config.json').read_text())
FC_T = {'ngram': tuple(bc['ngram']), 'char': bc['char_wb_2_5'], 'min_df': bc['min_df']}
LR_T = {'C': bc['C'], 'class_weight': bc['class_weight']}
print('chosen:', bc)

t0 = time.time()
_, Xtr_t, (Xva_t,) = featurise(FC_T, X_text[tr_idx], X_text[va_idx])
lr1 = make_clf('lr', LR_T).fit(Xtr_t, y[tr_idx])
p_lr1 = lr1.predict_proba(Xva_t)[:, 1]; s_lr1 = time.time() - t0
print(f'features: {Xtr_t.shape[1]:,}\n')
show_eval('LR tuned', p_lr1)

## 5. Preprocessing for the BiLSTM models

The BiLSTM reads **word sequences** instead of a bag of words. Its tokeniser is `preprocessing.tokenize`:
- lowercase and HTML-unescape the text
- keep contractions whole (`don't`, `isn't`) and keep `!` and `?` as tokens, since they carry sentiment
- no stop-word removal, so negations survive

In [ ]:
print('raw:            ', sample)
print('BiLSTM tokens:  ', tokenize(sample))

**Vocabulary and sequence length.** The vocabulary is built from the **training part only**. Words seen fewer than 2 times are mapped to `<unk>`: they are mostly typos, and the model can't learn anything reliable from one example. Index 0 is `<pad>`. Sequences are cut at **T_max = 256** tokens, which keeps about 99% of reviews whole. Batches are padded only to their longest review, and reviews of similar length are batched together, so very little time is spent on padding.

In [ ]:
MAX_LEN, MIN_FREQ, BATCH = 256, 2, 64

def build_vocab(texts, min_freq=MIN_FREQ):
    c = collections.Counter(t for s in texts for t in tokenize(s))
    vocab = {'<pad>': 0, '<unk>': 1}
    for w, n in c.most_common():
        if n >= min_freq:
            vocab[w] = len(vocab)
    return vocab, c

def encode(texts, vocab):
    return [[vocab.get(t, 1) for t in tokenize(s)][:MAX_LEN] or [1] for s in texts]

vocab_tr, counts_tr = build_vocab(X_text[tr_idx])
lens = np.array([len(tokenize(s)) for s in X_text[tr_idx]])
ids_va = encode(X_text[va_idx], vocab_tr)
unk_rate = np.mean([t == 1 for r in ids_va for t in r])
print(f'distinct tokens in training part: {len(counts_tr):,} -> vocabulary (min_freq={MIN_FREQ}): {len(vocab_tr):,}')
print(f'validation tokens mapped to <unk>: {unk_rate:.2%}')
print(f'review length (tokens): median {np.median(lens):.0f}, 95th pct {np.percentile(lens, 95):.0f}, '
      f'max {lens.max()}; truncated at {MAX_LEN}: {np.mean(lens > MAX_LEN):.2%} of reviews')

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(np.clip(lens, 0, 400), bins=np.arange(0, 410, 10), color=BLUE, edgecolor='white', linewidth=0.5)
ax.axvline(MAX_LEN, color=ORANGE, linestyle='--', linewidth=1.5, label=f'T_max = {MAX_LEN}')
ax.set_title('Tokens per training review (capped at 400)', loc='left', color=INK)
ax.set_xlabel('Tokens'); ax.set_ylabel('Reviews'); ax.legend(frameon=False); ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

**GloVe initialisation (variant B).** Each vocabulary word gets its 100-d GloVe vector [Pennington et al., 2014] (6B-token Wikipedia + Gigaword corpus). GloVe splits contractions (`don't` → `do` + `n't`), so a whole contraction gets the mean of its two pieces. Words GloVe doesn't know get small random vectors with the same spread as the GloVe vectors, and `<pad>` is all zeros. The vectors are a **starting point only**: they are fine-tuned along with the rest of the network.

In [ ]:
if not GLOVE.exists():
    GLOVE.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve('https://github.com/RaRe-Technologies/gensim-data/releases/download/'
                               'glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz', GLOVE)

def load_glove(needed):
    vecs = {}
    with gzip.open(GLOVE, 'rt', encoding='utf8') as f:
        next(f)                                         # header line: "400000 100"
        for line in f:
            w, rest = line.split(' ', 1)
            if w in needed:
                vecs[w] = np.asarray(rest.split(), dtype=np.float32)
    return vecs

def glove_pieces(w):
    if "'" not in w:
        return [w]
    if w.endswith("n't"):
        return [w[:-3], "n't"]
    a, b = w.split("'", 1)
    return [a, "'" + b]

all_vocab = build_vocab(X_text)[0]                      # superset, so one GloVe pass serves every split
needed = {p for w in all_vocab for p in glove_pieces(w)}
t0 = time.time(); GLOVE_VECS = load_glove(needed)
G_STD = float(np.std(np.stack(list(GLOVE_VECS.values()))))
print(f'loaded {len(GLOVE_VECS):,} GloVe vectors in {time.time() - t0:.0f}s (std {G_STD:.3f})')

def glove_matrix(vocab, seed):
    rng = np.random.default_rng(seed)
    E = rng.normal(0, G_STD, (len(vocab), 100)).astype(np.float32)
    E[0] = 0.0
    hit = 0
    for w, i in vocab.items():
        parts = [GLOVE_VECS[p] for p in glove_pieces(w) if p in GLOVE_VECS]
        if i > 1 and parts:
            E[i] = np.mean(parts, 0); hit += 1
    return torch.from_numpy(E), hit / (len(vocab) - 2)

_, cov = glove_matrix(vocab_tr, SEED)
tok_cov = np.mean([any(p in GLOVE_VECS for p in glove_pieces(t)) for s in X_text[tr_idx][:2000] for t in tokenize(s)])
print(f'GloVe covers {cov:.1%} of the vocabulary types and {tok_cov:.1%} of running tokens')

## 6. BiLSTM architecture

```
review text ──tokenize──▶ token ids x ∈ ℕ^(B×T)          (B = batch size, T ≤ 256 = longest review in batch)
  │
  ▼  Embedding(V, d)                 A: random d=128   B: GloVe d=100 (fine-tuned)        → (B, T, d)
  ▼  Dropout(p)
  ▼  BiLSTM(d → H per direction)     reads the review left→right and right→left           → (B, T, 2H)
  ├─▶ additive attention pooling     α_t = softmax_t( vᵀ tanh(W h_t) ),  c = Σ α_t h_t     → (B, 2H)
  └─▶ masked max-pooling             m = max_t h_t  (padding positions excluded)           → (B, 2H)
  ▼  concat [c ; m]                                                                         → (B, 4H)
  ▼  Dropout(p)  →  Linear(4H, 2)                                                           → logits (B, 2)
  ▼  softmax                                                                                → p(negative), p(positive)
```

**Why this design:**
- **BiLSTM:** reads the review in order, so it can model negation and contrast ("*not* comfortable", "looks nice *but* broke"). Bag-of-words TF-IDF can't see word order.
- **Attention pooling:** learns *which* words matter, so a review's sentiment can come from one sentence in a long review. The weights α can also be inspected (section 7).
- **Max-pooling:** keeps the strongest signal for each feature, which complements attention's weighted average.
- **Dropout:** regularises a model with about 1–2 M parameters trained on only about 6 k reviews.

In [ ]:
class BiLSTMAttn(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, dropout, pretrained=None, freeze=False):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if pretrained is not None:
            self.emb.weight.data.copy_(pretrained)
            self.emb.weight.requires_grad = not freeze
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.att_W = nn.Linear(2 * hidden, 2 * hidden)
        self.att_v = nn.Linear(2 * hidden, 1, bias=False)
        self.head = nn.Linear(4 * hidden, 2)

    def forward(self, x, lens, return_attn=False):
        e = self.drop(self.emb(x))                                               # (B, T, d)
        packed = pack_padded_sequence(e, lens, batch_first=True, enforce_sorted=False)
        h, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=x.size(1))  # (B, T, 2H)
        mask = torch.arange(x.size(1))[None, :] < lens[:, None]                  # (B, T) True = real token
        scores = self.att_v(torch.tanh(self.att_W(h))).squeeze(-1).masked_fill(~mask, -1e9)
        alpha = torch.softmax(scores, 1)                                         # (B, T)
        ctx = (alpha.unsqueeze(-1) * h).sum(1)                                   # (B, 2H)
        mx = h.masked_fill(~mask.unsqueeze(-1), -1e9).max(1).values              # (B, 2H)
        logits = self.head(self.drop(torch.cat([ctx, mx], 1)))                   # (B, 2)
        return (logits, alpha) if return_attn else logits

### Input/output dimensions and parameter counts

In [ ]:
def layer_table(model, x, lens):
    rows, hooks = [], []
    def hook(name):
        def f(m, i, o):
            o = o[0] if isinstance(o, tuple) else o
            shape = tuple(pad_packed_sequence(o, batch_first=True)[0].shape) if hasattr(o, 'batch_sizes') else tuple(o.shape)
            rows.append({'layer': name, 'module': m.__class__.__name__, 'output shape': str(shape),
                         'parameters': sum(p.numel() for p in m.parameters())})
        return f
    for name, m in model.named_children():
        hooks.append(m.register_forward_hook(hook(name)))
    model.eval()
    with torch.no_grad():
        model(x, lens)
    for h in hooks:
        h.remove()
    t = pd.DataFrame(rows).drop_duplicates('layer')
    total = sum(p.numel() for p in model.parameters())
    return pd.concat([t, pd.DataFrame([{'layer': 'TOTAL', 'module': '', 'output shape': '', 'parameters': total}])],
                     ignore_index=True)

xb = torch.randint(2, 100, (64, 120)); lb = torch.full((64,), 120)
print('Example batch: B=64 reviews, T=120 tokens, H=128, V =', len(vocab_tr))
print('\nVariant A (scratch, d=128):')
display(layer_table(BiLSTMAttn(len(vocab_tr), 128, 128, 0.4), xb, lb))
print('Variant B (GloVe, d=100):')
display(layer_table(BiLSTMAttn(len(vocab_tr), 100, 128, 0.4), xb, lb))

In [ ]:
# Architecture figure (for the report)
def arch_figure():
    steps = [('Raw review text', '"The zipper broke after 2 days, not worth it!"', GRAY),
             ('Tokenise + vocabulary lookup', 'token ids  (B × T),  T ≤ 256', GRAY),
             ('Embedding  V × d', 'A: random d=128  |  B: GloVe d=100, fine-tuned   →  (B × T × d)', BLUE),
             ('Dropout p', '', GRAY),
             ('Bidirectional LSTM  (H per direction)', 'forward + backward hidden states   →  (B × T × 2H)', BLUE),
             (None, None, None),
             ('Concatenate [c ; m]', '(B × 4H)', GRAY),
             ('Dropout p  →  Linear(4H → 2)', 'logits  (B × 2)', BLUE),
             ('Softmax', 'p(negative), p(positive)  →  label = argmax', AQUA)]
    fig, ax = plt.subplots(figsize=(9, 9.5)); ax.set_axis_off(); ax.set_xlim(0, 10); ax.set_ylim(0, len(steps) + 0.5)
    for k, (title, sub, col) in enumerate(steps):
        yc = len(steps) - k
        if title is None:
            for xc, t, s in [(2.6, 'Additive attention pooling', 'α = softmax(vᵀ tanh(W h))\nc = Σ α_t h_t   (B × 2H)'),
                             (7.4, 'Masked max-pooling', 'm = max_t h_t\n(B × 2H)')]:
                ax.add_patch(plt.Rectangle((xc - 2.2, yc - 0.38), 4.4, 0.76, facecolor='white', edgecolor=ORANGE, linewidth=1.6))
                ax.text(xc, yc + 0.12, t, ha='center', va='center', fontsize=10, weight='bold', color=INK)
                ax.text(xc, yc - 0.17, s, ha='center', va='center', fontsize=8, color=MUTED)
                ax.annotate('', (xc, yc + 0.4), (5, yc + 0.62), arrowprops=dict(arrowstyle='->', color=MUTED))
                ax.annotate('', (5, yc - 0.62), (xc, yc - 0.4), arrowprops=dict(arrowstyle='->', color=MUTED))
            continue
        ax.add_patch(plt.Rectangle((1.2, yc - 0.33), 7.6, 0.66, facecolor='white', edgecolor=col, linewidth=1.6))
        ax.text(5, yc + (0.1 if sub else 0), title, ha='center', va='center', fontsize=10, weight='bold', color=INK)
        if sub:
            ax.text(5, yc - 0.16, sub, ha='center', va='center', fontsize=8, color=MUTED)
        nxt = steps[k + 1][0] if k + 1 < len(steps) else 'end'
        if k + 1 < len(steps) and nxt is not None and title != 'Bidirectional LSTM  (H per direction)':
            ax.annotate('', (5, yc - 0.67), (5, yc - 0.33), arrowprops=dict(arrowstyle='->', color=MUTED))
    ax.set_title('BiLSTM + attention sentiment classifier', loc='left', color=INK, fontsize=12)
    plt.tight_layout()
    return fig

fig = arch_figure(); fig.savefig('../figures/bilstm_architecture.png', dpi=200, bbox_inches='tight'); plt.show()

## 7. Loss function and training strategy (BiLSTM)

| Item | Choice | Reason |
|---|---|---|
| Output | 2 logits → softmax | standard for 2-class classification; `argmax` = predicted label |
| **Loss** | **class-weighted cross-entropy**, w_c = N / (2·N_c) → w_neg ≈ 3.42, w_pos ≈ 0.59 | only 14.6% of reviews are negative. Without weights the model (like the LR) mostly predicts "positive". Weighting makes each class count equally |
| Optimiser | Adam, learning rate from the sweep (section 8) | adapts the step size per parameter; standard for LSTMs |
| Gradient clipping | max L2 norm 1.0 | prevents exploding gradients in recurrent networks |
| Batch size | 64, length-bucketed | small enough for regularising noise, large enough for speed; bucketing reduces padding |
| Epochs | up to 15, **early stopping** with patience 3 on validation macro-F1, best checkpoint restored | stops before overfitting; macro-F1 rather than accuracy, because accuracy rewards ignoring the minority class |
| Final model | refit on **all** of `train.json` for the median best epoch of the 3 seeds; test probabilities averaged over the 3 seeds | uses every labelled review; averaging seeds reduces variance |
| Reproducibility | fixed seeds (Python, NumPy, PyTorch, shuffling generator); CPU; results reported as mean ± std over seeds 42/43/44 and over 5 CV folds | sections 9 and 11 |

**Limitation:** the hold-out fold is used both for early stopping and for picking hyper-parameters, so its scores are slightly optimistic. The tuned TF-IDF model was also tuned with cross-validation on the same data, so the comparison stays fair. The real test is `test.json`, which is graded by the course.

In [ ]:
MAX_EPOCHS, PATIENCE = 15, 3
EMB_DIM = {'scratch': 128, 'glove': 100}

def batches(ids, gen, shuffle=True):
    n = len(ids)
    order = torch.randperm(n, generator=gen).tolist() if shuffle else list(range(n))
    chunk = BATCH * 50                                     # sort by length inside chunks -> little padding
    out = []
    for i in range(0, n, chunk):
        c = sorted(order[i:i + chunk], key=lambda j: len(ids[j]))
        out += [c[k:k + BATCH] for k in range(0, len(c), BATCH)]
    if shuffle:
        out = [out[i] for i in torch.randperm(len(out), generator=gen).tolist()]
    return out

def collate(ids, idx):
    L = torch.tensor([len(ids[j]) for j in idx])
    x = torch.zeros(len(idx), int(L.max()), dtype=torch.long)
    for r, j in enumerate(idx):
        x[r, :len(ids[j])] = torch.tensor(ids[j])
    return x, L

def predict_proba(model, ids):
    model.eval(); p = np.zeros(len(ids))
    with torch.no_grad():
        for idx in batches(ids, None, shuffle=False):
            x, L = collate(ids, idx)
            p[idx] = torch.softmax(model(x, L), 1)[:, 1].numpy()
    return p

def train_model(variant, cfg, tr_text, tr_y, va_text=None, va_y=None, seed=SEED, epochs=MAX_EPOCHS):
    """With validation data: early stopping on val macro-F1. Without: train exactly `epochs` epochs."""
    set_seed(seed)
    vocab, _ = build_vocab(tr_text)
    pre = glove_matrix(vocab, seed)[0] if variant == 'glove' else None
    model = BiLSTMAttn(len(vocab), EMB_DIM[variant], cfg['hidden'], cfg['dropout'], pre, cfg.get('freeze', False))
    w = torch.tensor(len(tr_y) / (2 * np.bincount(tr_y, minlength=2)), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=w if cfg.get('class_weight', True) else None)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=cfg['lr'])
    ids_tr, yt = encode(tr_text, vocab), torch.tensor(tr_y)
    ids_va = encode(va_text, vocab) if va_text is not None else None
    gen = torch.Generator().manual_seed(seed)
    best, best_state, best_ep, bad, hist, t0 = -1, None, epochs, 0, [], time.time()
    for ep in range(1, epochs + 1):
        model.train(); tot = 0.0
        for idx in batches(ids_tr, gen):
            x, L = collate(ids_tr, idx)
            opt.zero_grad()
            loss = loss_fn(model(x, L), yt[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tot += loss.item() * len(idx)
        rec = {'epoch': ep, 'train_loss': tot / len(ids_tr)}
        if ids_va is not None:
            p = predict_proba(model, ids_va)
            m = metrics(va_y, (p >= 0.5).astype(int), p)
            rec.update(val_loss=float(nn.functional.cross_entropy(torch.log(torch.tensor(np.c_[1 - p, p]).clamp_min(1e-7)),
                                                                   torch.tensor(va_y), weight=w.double()).item()),
                       val_macro_f1=m['macro_f1'], val_acc=m['accuracy'])
            if m['macro_f1'] > best:
                best, best_ep, bad = m['macro_f1'], ep, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
        hist.append(rec)
        if ids_va is not None and bad >= PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, vocab, best_ep, hist, time.time() - t0

# ---- cache: every (variant, config, seed) run is stored, so the notebook re-runs quickly
RUNS = pickle.load(open(CACHE, 'rb')) if CACHE.exists() and not RUN_ALL else {}

def run(variant, cfg, seed=SEED):
    """Train on the train part, evaluate on the hold-out fold (cached)."""
    key = (variant, json.dumps(cfg, sort_keys=True), seed)
    if key not in RUNS:
        model, vocab, best_ep, hist, secs = train_model(variant, cfg, X_text[tr_idx], y[tr_idx],
                                                        X_text[va_idx], y[va_idx], seed=seed)
        p = predict_proba(model, encode(X_text[va_idx], vocab))
        RUNS[key] = {'variant': variant, **cfg, 'seed': seed, 'best_epoch': best_ep, 'train_seconds': round(secs, 1),
                     'n_params': sum(q.numel() for q in model.parameters()), 'p_val': p, 'history': hist,
                     **{k: v for k, v in metrics(y[va_idx], (p >= 0.5).astype(int), p).items()}}
        pickle.dump(RUNS, open(CACHE, 'wb'))
        r = RUNS[key]
        print(f"  trained {variant:7s} {cfg} seed {seed}: macro-F1 {r['macro_f1']:.4f} acc {r['accuracy']:.4f} "
              f"(best epoch {best_ep}, {secs:.0f}s)")
    return RUNS[key]

## 8. BiLSTM hyper-parameter selection

**Method:** a full grid (4 × 3 × 3 × 2 = 72 runs per variant) would take hours, so parameters are tuned **one at a time** in order of expected impact. Each step keeps the best value found so far. Every run uses seed 42 and the same hold-out fold, and is **scored on macro-F1**.

| Step | Parameter | Values tried | Why this range |
|---|---|---|---|
| start | | lr 1e-3, H 128, dropout 0.4, weighted loss | Adam's usual default; mid-size LSTM; moderate dropout |
| 1 | learning rate | 5e-4, 1e-3, 2e-3, 3e-3 | the most sensitive setting: too small trains slowly or stops early, too large is unstable |
| 2 | hidden size H | 64, 128, 256 | model capacity against overfitting on about 6 k reviews |
| 3 | dropout p | 0.2, 0.4, 0.6 | regularisation strength |
| 4 | class-weighted loss | on, off | does weighting really help the minority class? |
| 5 (GloVe only) | freeze embeddings | fine-tune, frozen | is adapting GloVe to product reviews worth it? |

Fixed by design rather than swept: embedding size (128 scratch / 100 GloVe), batch size 64, T_max 256, min_freq 2, 1 LSTM layer. A 2nd layer adds parameters that 6 k reviews can't support.

In [ ]:
SWEEP = {'lr': [5e-4, 1e-3, 2e-3, 3e-3], 'hidden': [64, 128, 256], 'dropout': [0.2, 0.4, 0.6], 'class_weight': [True, False]}
BASE = {'lr': 1e-3, 'hidden': 128, 'dropout': 0.4, 'class_weight': True}

sweep_rows, BEST = [], {}
for variant in ['scratch', 'glove']:
    cfg = dict(BASE)
    steps = list(SWEEP.items()) + ([('freeze', [False, True])] if variant == 'glove' else [])
    for step, (param, values) in enumerate(steps, 1):
        res = []
        for v in values:
            c = {**cfg, param: v}
            if param != 'freeze':
                c.pop('freeze', None)
            r = run(variant, c)
            res.append((r['macro_f1'], v))
            sweep_rows.append({'variant': variant, 'step': step, 'param': param, 'value': v,
                               **{k: r[k] for k in ['macro_f1', 'accuracy', 'recall_neg', 'roc_auc', 'best_epoch', 'train_seconds']}})
        cfg[param] = max(res, key=lambda t: t[0])[1]
    if not cfg.get('freeze'):
        cfg.pop('freeze', None)
    BEST[variant] = cfg

sweep = pd.DataFrame(sweep_rows)
sweep.to_csv(RESULTS / 'bilstm_sweep.csv', index=False)
for v in BEST:
    print(f'best {v:7s}: {BEST[v]}')

In [ ]:
params = ['lr', 'hidden', 'dropout', 'class_weight']
metric_rows = [('macro_f1', 'Macro-F1\n(selection metric)'), ('accuracy', 'Overall accuracy'),
               ('recall_neg', 'Negative-review\naccuracy')]
fig, axes = plt.subplots(3, 4, figsize=(13, 8.5), sharey='row')
for r, (met, ylab) in enumerate(metric_rows):
    for ax, prm in zip(axes[r], params):
        for variant in ['scratch', 'glove']:
            s = sweep[(sweep.variant == variant) & (sweep.param == prm)]
            xs = [str(v) for v in s.value]
            ax.plot(xs, s[met], 'o-', color=COL[variant], linewidth=2, markersize=7,
                    label='A: scratch' if variant == 'scratch' else 'B: GloVe')
            b = s.loc[s.macro_f1.idxmax()]
            ax.plot(str(b.value), b[met], 'o', markersize=12, markerfacecolor='none', markeredgecolor=COL[variant], markeredgewidth=1.5)
        if r == 0:
            ax.set_title({'lr': 'Step 1: learning rate', 'hidden': 'Step 2: hidden size H', 'dropout': 'Step 3: dropout',
                          'class_weight': 'Step 4: weighted loss'}[prm], loc='left', color=INK, fontsize=10)
        ax.grid(axis='x', visible=False)
    axes[r][0].set_ylabel(ylab)
axes[0][0].legend(frameon=False, loc='lower left')
fig.suptitle('One-factor-at-a-time sweep (seed 42); ring = value kept (chosen by macro-F1)', x=0.01, ha='left', color=INK)
plt.tight_layout(); plt.show()

show = sweep.rename(columns={'recall_neg': 'neg_review_accuracy'}).copy(); show['value'] = show['value'].astype(str)
show[['variant', 'step', 'param', 'value', 'accuracy', 'neg_review_accuracy', 'macro_f1', 'roc_auc', 'best_epoch', 'train_seconds']].round(4)

### How each BiLSTM setting was chosen (part c)

| Setting | Scratch (A) | GloVe (B) | Reasoning from the sweep |
|---|---|---|---|
| **Learning rate** | **2e-3** | **1e-3** | 5e-4 is too slow: early stopping ends training before the model has converged (macro-F1 0.78 / 0.82). At 3e-3 training becomes unstable, and GloVe peaks at epoch 2. Random embeddings must move far from their start, so they need a larger step. GloVe vectors are already meaningful, so a smaller step avoids destroying them. |
| **Hidden size H** | **64** | **256** | With random embeddings, extra capacity only overfits (H=256: 0.844 < H=64: 0.850). GloVe gives the LSTM richer inputs, so the larger H=256 can use them (0.852). |
| **Dropout** | **0.4** | **0.6** | 0.2 overfits in both variants (lowest scores). The larger GloVe model (H=256, about 1.7 M parameters) needs stronger regularisation, so 0.6 is best for it. |
| **Class-weighted loss** | **on** | **off** | Scratch: weighting improves macro-F1 (0.850 vs 0.842) and negative-review accuracy (75% vs 72%). GloVe: macro-F1 is higher **without** weighting (0.865 vs 0.856), but negative-review accuracy falls from 79% to 72%. The sweep optimises macro-F1, so weighting was switched off. Section 11 shows the cost of that choice. |
| **Freeze GloVe** | n/a | **fine-tune** | Frozen GloVe scores much worse (0.794 vs 0.865, negative accuracy 55%). General-purpose vectors don't separate product-review sentiment well. For example, "cheap" is neutral in Wikipedia text but negative in reviews, so the vectors must adapt. |

**Takeaway:** the learning rate and freezing matter most, with spreads of 0.07–0.08 macro-F1. Hidden size and dropout change macro-F1 by only about 0.01–0.02, which is close to the seed-to-seed noise (std ≈ 0.01, section 9). So their exact values shouldn't be over-interpreted.

## 9. Reproducibility: 3 seeds per variant

The best configuration of each variant is retrained with seeds **42, 43 and 44**. The seed controls weight initialisation, dropout masks, batch order and the random vectors for words GloVe doesn't know. Results are reported as **mean ± standard deviation**. Seed 42 was already trained in the sweep. Retraining it from scratch and getting **identical** numbers shows that a fixed seed reproduces a run exactly.

In [ ]:
seed_rows = []
for variant, cfg in BEST.items():
    for seed in SEEDS:
        r = run(variant, cfg, seed)
        seed_rows.append({'variant': variant, 'seed': seed, **{k: r[k] for k in
                          ['accuracy', 'macro_f1', 'recall_neg', 'f1_neg', 'roc_auc', 'best_epoch', 'train_seconds', 'n_params']}})
seeds_df = pd.DataFrame(seed_rows)
summary = seeds_df.rename(columns={'recall_neg': 'neg_review_accuracy'}).groupby('variant')[['accuracy', 'neg_review_accuracy', 'macro_f1', 'roc_auc']].agg(['mean', 'std']).round(4)
display(seeds_df.round(4)); display(summary)

# determinism check: retrain seed 42 of the best scratch config without the cache
v0 = min(BEST, key=lambda v: RUNS[(v, json.dumps(BEST[v], sort_keys=True), SEED)]['train_seconds'])
m_re, voc_re, *_ = train_model(v0, BEST[v0], X_text[tr_idx], y[tr_idx], X_text[va_idx], y[va_idx], seed=SEED)
p_re = predict_proba(m_re, encode(X_text[va_idx], voc_re))
p_cached = RUNS[(v0, json.dumps(BEST[v0], sort_keys=True), SEED)]['p_val']
print(f'Determinism check ({v0}, seed 42 retrained): max |Δ p| = {np.abs(p_re - p_cached).max():.2e}, '
      f'identical predictions: {np.array_equal(p_re >= 0.5, p_cached >= 0.5)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for variant in BEST:
    for seed in SEEDS:
        h = pd.DataFrame(RUNS[(variant, json.dumps(BEST[variant], sort_keys=True), seed)]['history'])
        lab = ('A: scratch' if variant == 'scratch' else 'B: GloVe') if seed == SEED else None
        axes[0].plot(h.epoch, h.train_loss, '-', color=COL[variant], linewidth=2, alpha=0.9, label=lab)
        axes[0].plot(h.epoch, h.val_loss, '--', color=COL[variant], linewidth=1.5, alpha=0.9)
        axes[1].plot(h.epoch, h.val_macro_f1, 'o-', color=COL[variant], linewidth=2, markersize=5, alpha=0.9, label=lab)
        b = h.val_macro_f1.idxmax()
        axes[1].plot(h.epoch[b], h.val_macro_f1[b], '*', color=COL[variant], markersize=14)
axes[0].set_title('Weighted cross-entropy (solid = train, dashed = validation)', loc='left', color=INK, fontsize=10)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(frameon=False)
axes[1].set_title('Validation macro-F1 per epoch (★ = checkpoint kept)', loc='left', color=INK, fontsize=10)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro-F1'); axes[1].legend(frameon=False, loc='lower right')
for ax in axes: ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

**Reading the curves:** training loss keeps falling while validation loss reaches its minimum after a few epochs and then rises. That is classic overfitting, and early stopping on validation macro-F1 keeps the checkpoint from before it sets in (★).

## 10. Hold-out comparison of all four models

All models are scored on the **same 1,480 validation reviews** (fold 0). BiLSTM rows show the mean ± std over 3 seeds, plus the **3-seed ensemble** (averaged probabilities) that is used for the test predictions.

The key question is **how many of the 216 negative reviews each model gets right**: Part 1's LR gets only 28% of them.

In [ ]:
def row(name, p, params, secs, std=None):
    m = metrics(y[va_idx], (p >= 0.5).astype(int), p)
    r = {'model': name, 'accuracy': m['accuracy'], 'macro_f1': m['macro_f1'], 'recall_neg': m['recall_neg'],
         'precision_neg': m['precision_neg'], 'roc_auc': m['roc_auc'], 'parameters': params, 'train_s': round(secs, 1)}
    if std is not None:
        r.update({k: f"{seeds_df[seeds_df.variant == std][k].mean():.4f} ± {seeds_df[seeds_df.variant == std][k].std():.4f}"
                  for k in ['accuracy', 'macro_f1', 'recall_neg', 'roc_auc']})
    return r

P = {'LR (default)': p_lr0, 'LR (tuned)': p_lr1}
rows = [row('TF-IDF + LR (default, Part 1)', p_lr0, lr0.coef_.size + 1, s_lr0),
        row('TF-IDF + LR (tuned)', p_lr1, lr1.coef_.size + 1, s_lr1)]
for variant, name in [('scratch', 'BiLSTM-attn A: scratch'), ('glove', 'BiLSTM-attn B: GloVe')]:
    rs = [RUNS[(variant, json.dumps(BEST[variant], sort_keys=True), s)] for s in SEEDS]
    rows.append(row(name + ' (3 seeds, mean ± std)', rs[0]['p_val'], rs[0]['n_params'],
                    np.mean([r['train_seconds'] for r in rs]), std=variant))
    P[f'{variant} ensemble'] = np.mean([r['p_val'] for r in rs], 0)
    rows.append(row(name + ' (3-seed ensemble)', P[f'{variant} ensemble'], rs[0]['n_params'],
                    sum(r['train_seconds'] for r in rs)))
comp = pd.DataFrame(rows)
comp.to_csv(RESULTS / 'model_comparison.csv', index=False)
print('all four models scored on the same', len(va_idx), 'validation reviews')

In [ ]:
short = ['LR default (Part 1)', 'LR tuned', 'BiLSTM A: scratch', 'BiLSTM B: GloVe']
keys = ['LR (default)', 'LR (tuned)', 'scratch ensemble', 'glove ensemble']
cols = [GRAY, AQUA, ORANGE, BLUE]
n_neg, n_pos = int((y[va_idx] == 0).sum()), int((y[va_idx] == 1).sum())
head = []
for name, k in zip(short, keys):
    pred = (P[k] >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[va_idx], pred).ravel()
    head.append({'model': name, 'overall accuracy': (tn + tp) / len(pred),
                 'negative reviews correct': f'{tn} / {n_neg}', 'negative accuracy': tn / n_neg,
                 'positive reviews correct': f'{tp} / {n_pos}', 'positive accuracy': tp / n_pos,
                 'macro-F1': f1_score(y[va_idx], pred, average='macro'), '_tn': tn})
head = pd.DataFrame(head)
display(head.drop(columns='_tn').style.format({c: '{:.1%}' for c in ['overall accuracy', 'negative accuracy', 'positive accuracy']}
                                              | {'macro-F1': '{:.3f}'}).hide(axis='index'))
base = head.iloc[0]
for i in [2, 3]:
    r = head.iloc[i]
    print(f"{r.model}: overall accuracy {r['overall accuracy']:.1%} vs {base['overall accuracy']:.1%} for Part 1 LR "
          f"({(r['overall accuracy'] - base['overall accuracy']) * 100:+.1f} pts); negative reviews correct "
          f"{r._tn}/{n_neg} vs {base._tn}/{n_neg} ({r._tn - base._tn:+d} reviews, {r['negative accuracy']:.0%} vs {base['negative accuracy']:.0%})")

fig, ax = plt.subplots(figsize=(11, 4.2))
groups = [('overall accuracy', 'Overall accuracy'), ('negative accuracy', 'Negative reviews (216)'), ('positive accuracy', 'Positive reviews (1264)')]
w = 0.19
for j, (name, col) in enumerate(zip(short, cols)):
    vals = head.iloc[j][[g for g, _ in groups]].astype(float).values
    xs = np.arange(len(groups)) + (j - 1.5) * (w + 0.01)
    bars = ax.bar(xs, vals, width=w, color=col, label=name)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f'{v:.0%}', ha='center', color=INK, fontsize=8.5)
ax.set_xticks(np.arange(len(groups)), [t for _, t in groups]); ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_ylabel('Share classified correctly'); ax.grid(axis='x', visible=False)
ax.set_title('Validation accuracy per class: the gap is in the negative reviews', loc='left', color=INK)
ax.legend(frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.1))
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, name, key in zip(axes, short, ['LR (default)', 'LR (tuned)', 'scratch ensemble', 'glove ensemble']):
    cm = confusion_matrix(y[va_idx], (P[key] >= 0.5).astype(int))
    ax.imshow(cm, cmap='Blues', vmin=0); ax.grid(False)
    for (r, c), v in np.ndenumerate(cm):
        ax.text(c, r, f'{v}\n{v / cm[r].sum():.0%}', ha='center', va='center', color='white' if v > cm.max() / 2 else INK, fontsize=9)
    ax.set_xticks([0, 1], ['pred neg', 'pred pos']); ax.set_yticks([0, 1], ['neg', 'pos'])
    ax.set_title(name, loc='left', color=INK, fontsize=10)
plt.tight_layout(); plt.show()
print('Full comparison (BiLSTM mean ± std over 3 seeds, and 3-seed ensemble):')
comp.round(4)

In [ ]:
from sklearn.metrics import roc_auc_score
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, key, col in zip(short, keys, cols):
    fpr, tpr, _ = roc_curve(y[va_idx], P[key])
    ax.plot(fpr, tpr, color=col, linewidth=2, label=f'{name} (AUC {roc_auc_score(y[va_idx], P[key]):.3f})')
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle='--', label='random guess')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC curves on the hold-out fold', loc='left', color=INK); ax.legend(frameon=False, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

### Hold-out result
- **Both BiLSTMs fix Part 1's main weakness.** On the same 1,480 reviews, the Part 1 LR gets 61/216 negative reviews right (28%). BiLSTM-scratch gets 157/216 (73%) and BiLSTM-GloVe gets 163/216 (75%). Overall accuracy rises from 89.3% to 92.4% and 93.4% (3-seed ensembles).
- **BiLSTM-GloVe is the best model on this split:** highest accuracy (93.4%), highest macro-F1 (0.866) and the most precise negative predictions (79% of its "negative" calls are right, against 70% for the tuned LR).
- The **tuned LR** is a strong baseline. It matches GloVe on negative reviews (163/216) but makes more false-negative calls on positive reviews, so its overall accuracy is 91.7%.
- **Caveat:** the BiLSTM settings were chosen on this fold. Section 11 checks whether the advantage holds on folds that played no part in tuning.

## 11. 5-fold cross-validation of all four models

A single hold-out split can be lucky or unlucky. Here **every model is trained and scored 5 times**, once with each fold as the validation set. The same grouped, stratified folds are used throughout, and each review is a validation review exactly once.
- **TF-IDF models:** the vectorizer and LR are refitted on the 4 training folds every time.
- **BiLSTM models:** the chosen configuration from section 8 is retrained from scratch on the 4 training folds (seed 42), with early stopping on the held-out fold, as in the hold-out protocol. Fold 0 is the run from section 9, and folds 1–4 are new runs, also cached.

Hyper-parameters were chosen on fold 0, so fold 0 is slightly optimistic for the BiLSTMs. Folds 1–4 are untouched by any tuning decision.

In [ ]:
def bilstm_fold(variant, cfg, k, seed=SEED):
    if k == 0:
        return run(variant, cfg, seed)['p_val']
    key = ('cv', variant, json.dumps(cfg, sort_keys=True), k, seed)
    if key not in RUNS:
        tr, va = folds[k]
        model, vocab, best_ep, hist, secs = train_model(variant, cfg, X_text[tr], y[tr], X_text[va], y[va], seed=seed)
        RUNS[key] = {'p_val': predict_proba(model, encode(X_text[va], vocab)), 'best_epoch': best_ep, 'train_seconds': secs}
        pickle.dump(RUNS, open(CACHE, 'wb'))
        print(f'  trained {variant} fold {k}: best epoch {best_ep}, {secs:.0f}s')
    return RUNS[key]['p_val']

CV_MODELS = ['LR default (Part 1)', 'LR tuned', 'BiLSTM A: scratch', 'BiLSTM B: GloVe']
cv_rows = []
for k, (tr, va) in enumerate(folds):
    v = TfidfVectorizer().fit(X_text[tr])
    p0 = LogisticRegression(max_iter=1000, random_state=SEED).fit(v.transform(X_text[tr]), y[tr]).predict_proba(v.transform(X_text[va]))[:, 1]
    _, A, (B,) = featurise(FC_T, X_text[tr], X_text[va])
    p1 = make_clf('lr', LR_T).fit(A, y[tr]).predict_proba(B)[:, 1]
    p2 = bilstm_fold('scratch', BEST['scratch'], k)
    p3 = bilstm_fold('glove', BEST['glove'], k)
    for name, p in zip(CV_MODELS, [p0, p1, p2, p3]):
        a = class_accuracy(y[va], p)
        cv_rows.append({'fold': k, 'model': name, 'n_val': len(va), 'n_neg': int((y[va] == 0).sum()),
                        **{m: a[m] for m in ['accuracy', 'neg_accuracy', 'pos_accuracy', 'macro_f1', 'neg_correct']}})
cvdf = pd.DataFrame(cv_rows)
cvdf.to_csv(RESULTS / 'cv5_all_models.csv', index=False)

agg = cvdf.groupby('model', sort=False)[['accuracy', 'neg_accuracy', 'pos_accuracy', 'macro_f1']].agg(['mean', 'std'])
cv_summary = pd.DataFrame({m: [f"{agg.loc[n, (m, 'mean')]:.2%} ± {agg.loc[n, (m, 'std')]:.2%}" if m != 'macro_f1'
                               else f"{agg.loc[n, (m, 'mean')]:.4f} ± {agg.loc[n, (m, 'std')]:.4f}" for n in CV_MODELS]
                           for m in ['accuracy', 'neg_accuracy', 'pos_accuracy', 'macro_f1']}, index=CV_MODELS)
cv_summary.columns = ['accuracy', 'negative-review accuracy', 'positive-review accuracy', 'macro-F1']
print('5-fold cross-validation, mean ± std over folds:')
display(cv_summary)
print('Per-fold accuracy:')
display(cvdf.pivot(index='fold', columns='model', values='accuracy')[CV_MODELS].style.format('{:.2%}'))
print('Per-fold negative reviews correct:')
display(cvdf.pivot(index='fold', columns='model', values='neg_correct')[CV_MODELS])

In [ ]:
cols4 = dict(zip(CV_MODELS, [GRAY, AQUA, ORANGE, BLUE]))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (m, title) in zip(axes, [('accuracy', 'Overall accuracy'), ('neg_accuracy', 'Negative-review accuracy'),
                                 ('macro_f1', 'Macro-F1')]):
    for i, name in enumerate(CV_MODELS):
        vals = cvdf[cvdf.model == name][m].values
        ax.scatter(np.full(len(vals), i) + np.linspace(-0.12, 0.12, len(vals)), vals, s=36, color=cols4[name],
                   edgecolor='white', linewidth=1, zorder=3)
        ax.hlines(vals.mean(), i - 0.28, i + 0.28, color=cols4[name], linewidth=2.5, zorder=4)
        ax.text(i + 0.3, vals.mean(), f'{vals.mean():.1%}' if m != 'macro_f1' else f'{vals.mean():.3f}',
                va='center', fontsize=8.5, color=INK)
    ax.set_xticks(range(4), ['LR\ndefault', 'LR\ntuned', 'BiLSTM\nscratch', 'BiLSTM\nGloVe'])
    ax.set_xlim(-0.5, 3.8); ax.grid(axis='x', visible=False)
    if m != 'macro_f1':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_title(title, loc='left', color=INK, fontsize=10)
fig.suptitle('5-fold cross-validation: dots = folds, bar = mean', x=0.01, ha='left', color=INK)
plt.tight_layout(); plt.show()

g, t = cvdf[cvdf.model == 'BiLSTM B: GloVe'].set_index('fold'), cvdf[cvdf.model == 'LR tuned'].set_index('fold')
d = cvdf[cvdf.model == 'LR default (Part 1)'].set_index('fold')
print(f"BiLSTM-GloVe beats LR default on accuracy in {(g.accuracy > d.accuracy).sum()}/5 folds "
      f"(mean +{(g.accuracy - d.accuracy).mean() * 100:.1f} pts) and on negative-review accuracy in {(g.neg_accuracy > d.neg_accuracy).sum()}/5 folds "
      f"(mean +{(g.neg_accuracy - d.neg_accuracy).mean() * 100:.1f} pts).")
print(f"BiLSTM-GloVe vs LR tuned: accuracy higher in {(g.accuracy > t.accuracy).sum()}/5 folds "
      f"(mean {(g.accuracy - t.accuracy).mean() * 100:+.1f} pts); macro-F1 higher in {(g.macro_f1 > t.macro_f1).sum()}/5 folds "
      f"(mean {(g.macro_f1 - t.macro_f1).mean():+.4f}).")

### What cross-validation shows
- **Against Part 1 (LR default), the improvement is robust.** BiLSTM-GloVe is more accurate in **5/5 folds** (88.9% → 92.8% on average) and gets far more negative reviews right in every fold (27% → 67% on average). BiLSTM-scratch improves similarly (91.7%, 71% negatives).
- **Against the tuned LR, it is a tie on accuracy and a loss on negatives.** Average accuracy is 92.8% (GloVe) vs 92.6% (tuned LR), a gap smaller than either model's fold-to-fold std (about 0.7%). The tuned LR finds **more** negative reviews (76.5% vs 67.5%) and has a higher macro-F1 (0.855 vs 0.845).
- **Why the hold-out result looked better:** fold 0 is the split the BiLSTM was tuned on, so its 93.4% is optimistic. Also, the chosen GloVe configuration has **no class weighting**, which favoured macro-F1 on fold 0. On new folds that shows up as lower negative-review accuracy (fold 4: 129/216). The tuned LR uses balanced class weights and was tuned by 5-fold CV, so its negative-review accuracy is more stable (std 1.7% vs 5.4%).
- **The BiLSTMs vary more between folds** (negative accuracy std 5–8% vs 1.7%). With only about 1,080 negative training reviews, a 1–1.7 M-parameter network is sensitive to which reviews it sees. A linear model with about 90 k features is much more stable.

**Conclusion for the report:** our own neural model clearly beats the Part 1 baseline and **matches** a carefully tuned TF-IDF model on accuracy, but it does not beat it on negative reviews. The most promising next steps:
- put class weighting back for the GloVe model, or tune the decision threshold
- choose hyper-parameters by cross-validation instead of on one fold
- average more seeds or folds

## 12. Error analysis: where the BiLSTM and the default LR disagree

Attention weights α show which tokens the BiLSTM relied on. The top-5 attended tokens are listed for each review.

In [ ]:
best_variant = max(BEST, key=lambda v: comp.set_index('model').loc[
    f"BiLSTM-attn {'A: scratch' if v == 'scratch' else 'B: GloVe'} (3-seed ensemble)", 'macro_f1'])
cfg_b = BEST[best_variant]
m_att, voc_att, *_ = train_model(best_variant, cfg_b, X_text[tr_idx], y[tr_idx], X_text[va_idx], y[va_idx], seed=SEED)
inv = {i: w for w, i in voc_att.items()}

def top_attention(text, k=5):
    toks = tokenize(text)[:MAX_LEN] or ['<unk>']
    ids = [voc_att.get(t, 1) for t in toks]
    m_att.eval()
    with torch.no_grad():
        logits, a = m_att(torch.tensor([ids]), torch.tensor([len(ids)]), return_attn=True)
    a = a[0].numpy()
    return ', '.join(f'{toks[i]}({a[i]:.2f})' for i in np.argsort(-a)[:k])

pb = P[f'{best_variant} ensemble']
ev = pd.DataFrame({'review': X_text[va_idx], 'actual': y[va_idx], 'LR_default_p': p_lr0.round(3), 'BiLSTM_p': pb.round(3)})
ev['LR_ok'] = (p_lr0 >= 0.5) == ev.actual
ev['BiLSTM_ok'] = (pb >= 0.5) == ev.actual
print(pd.crosstab(ev.LR_ok, ev.BiLSTM_ok, rownames=['LR correct'], colnames=['BiLSTM correct']))
print(f'\nBiLSTM ({best_variant}) right, LR wrong:')
a = ev[ev.BiLSTM_ok & ~ev.LR_ok].sort_values('BiLSTM_p').head(6).copy()
a['top attention'] = a.review.map(top_attention)
display(a.drop(columns=['LR_ok', 'BiLSTM_ok']))

In [ ]:
print('BiLSTM wrong (most confident mistakes):')
b = ev[~ev.BiLSTM_ok].assign(conf=lambda d: (d.BiLSTM_p - 0.5).abs()).sort_values('conf', ascending=False).head(6).copy()
b['top attention'] = b.review.map(top_attention)
display(b.drop(columns=['LR_ok', 'BiLSTM_ok', 'conf']))

## 13. Test predictions (`test.json`)

Every model is refitted on **all 7,401 training reviews** and predicts `test.json`:
- the TF-IDF models with their fixed settings
- each BiLSTM for the median best epoch of its 3 seed runs, once per seed, with the test probabilities averaged over the 3 seeds

`test.json` has no labels, so the predicted class split is shown instead of an accuracy. The model with the best hold-out ensemble macro-F1 is written to `submission_bilstm.csv`.

In [ ]:
def bilstm_test(variant):
    cfg = BEST[variant]
    eps = [RUNS[(variant, json.dumps(cfg, sort_keys=True), s)]['best_epoch'] for s in SEEDS]
    n_ep = int(np.median(eps))
    probs = []
    for seed in SEEDS:
        key = ('final', variant, json.dumps(cfg, sort_keys=True), seed, n_ep)
        if key not in RUNS:
            model, vocab, *_, secs = train_model(variant, cfg, X_text, y, seed=seed, epochs=n_ep)
            RUNS[key] = {'p_test': predict_proba(model, encode(test_df['reviews'].values, vocab)), 'secs': secs}
            pickle.dump(RUNS, open(CACHE, 'wb'))
            print(f'  refit {variant} seed {seed}: {secs:.0f}s')
        probs.append(RUNS[key]['p_test'])
    print(f'{variant}: config {cfg}, best epochs per seed {eps} -> refit on all 7,401 reviews for {n_ep} epochs x 3 seeds')
    return np.mean(probs, 0)

TEST_P = {'LR default (Part 1)': LogisticRegression(max_iter=1000, random_state=SEED).fit(vec_full.transform(X_text), y)
                                   .predict_proba(vec_full.transform(test_df['reviews']))[:, 1]}
_, A, (B,) = featurise(FC_T, X_text, test_df['reviews'].values)
TEST_P['LR tuned'] = make_clf('lr', LR_T).fit(A, y).predict_proba(B)[:, 1]
TEST_P['BiLSTM A: scratch'] = bilstm_test('scratch')
TEST_P['BiLSTM B: GloVe'] = bilstm_test('glove')

p_test = TEST_P[{'scratch': 'BiLSTM A: scratch', 'glove': 'BiLSTM B: GloVe'}[best_variant]]
pred_test = (p_test >= 0.5).astype(int)
sub = pd.DataFrame({'reviews': test_df['reviews'].values, 'predictions': pred_test})
sub.to_csv(RESULTS / 'submission_bilstm.csv', index=False)
print(f'\nsubmission (best model, {best_variant}) saved to {RESULTS / "submission_bilstm.csv"} {sub.shape}')

tp = pd.DataFrame({n: (p >= 0.5).astype(int) for n, p in TEST_P.items()})
print('\nPredicted class split on test.json (train.json is 85.4% positive):')
display(pd.DataFrame({'predicted negative': (tp == 0).sum(), 'predicted positive': (tp == 1).sum(),
                      'positive rate': tp.mean().map('{:.1%}'.format),
                      'agreement with submitted model': tp.apply(lambda c: f'{np.mean(c == pred_test):.1%}')}))

fig, ax = plt.subplots(figsize=(10, 3.4))
neg = (tp == 0).sum().values
bars = ax.barh(list(TEST_P)[::-1], neg[::-1], color=[GRAY, AQUA, ORANGE, BLUE][::-1], height=0.6)
for b, v in zip(bars, neg[::-1]):
    ax.text(v + 4, b.get_y() + b.get_height() / 2, f'{v} ({v / len(tp):.1%})', va='center', color=INK, fontsize=9)
ax.set_title('Test reviews predicted negative by each model (of 1,851)', loc='left', color=INK); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
sub.assign(p_positive=p_test.round(3)).head(10)

## 14. Summary

| Model | Hold-out accuracy | Hold-out negative reviews correct | 5-fold CV accuracy | 5-fold CV negative-review accuracy | 5-fold CV macro-F1 |
|---|---|---|---|---|---|
| 1. TF-IDF + LR default (Part 1) | 89.3% | 61/216 (28%) | 88.9% ± 0.5% | 27.2% ± 2.2% | 0.677 ± 0.016 |
| 2. TF-IDF + LR tuned | 91.7% | 163/216 (75%) | 92.6% ± 0.7% | **76.5% ± 1.7%** | **0.855 ± 0.011** |
| 3. BiLSTM + attention, scratch | 92.4%* | 157/216 (73%)* | 91.7% ± 0.9% | 71.4% ± 8.0% | 0.834 ± 0.016 |
| 4. BiLSTM + attention, GloVe | **93.4%*** | 163/216 (75%)* | **92.8% ± 0.7%** | 67.5% ± 5.4% | 0.845 ± 0.019 |

\* 3-seed ensemble on fold 0, the fold the BiLSTM settings were tuned on. The CV columns use seed 42 on all 5 folds.

**Key points**
- **Preprocessing:** TF-IDF follows the example notebook (default `TfidfVectorizer`) and, for the tuned model, adds word bigrams and character 2–5-grams. The BiLSTMs use a sentiment-aware tokeniser that keeps negations, contractions and `!`/`?`, a training-only vocabulary (min_freq 2) and sequences of at most 256 tokens.
- **Our model (part b):** Embedding → BiLSTM → attention pooling + max-pooling → Linear(4H→2). Trained with Adam, cross-entropy (class-weighted for scratch), gradient clipping and early stopping on macro-F1. Reproducible: fixed seeds on CPU, verified bit-identical, and reported over 3 seeds and 5 folds.
- **Parameter choice (part c):** a one-factor-at-a-time sweep of learning rate, hidden size, dropout, class weighting and freezing, scored on macro-F1 (section 8).
- **Result:** every model after Part 1 fixes the negative-review problem, getting roughly 2.5–2.8× as many negatives right. BiLSTM-GloVe has the highest accuracy, while the tuned TF-IDF LR is the most reliable on negative reviews.
- **Submission:** `../../results/submission_bilstm.csv` (BiLSTM-GloVe, 3-seed ensemble). Section 13 also gives every model's test predictions.